<a href="https://colab.research.google.com/github/shreyaghora/Bank_Marketing_Project/blob/main/Code/Smote.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install lightgbm
!pip install catboost
!pip install xgboost
!pip install specificity
!pip install imbalanced-learn

!pip install ipython-autotime
%load_ext autotime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.6 MB/s eta 0:00:00
time: 304 µs (started: 2026-04-14 07:07:20 +00:00)


In [2]:
import numpy as np
import pandas as pd
import time
from scipy.stats import uniform


from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, BaggingClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef, cohen_kappa_score, confusion_matrix, balanced_accuracy_score

time: 8.09 s (started: 2026-04-14 07:07:52 +00:00)


In [3]:
def specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

def gmean(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn)
    spec = tn / (tn + fp)
    return np.sqrt(sensitivity * spec)

def fpr_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return fp / (fp + tn) if (fp + tn) != 0 else 0

time: 1.06 ms (started: 2026-04-14 07:08:30 +00:00)


In [7]:
df = pd.read_csv("/content/bank-additional-full.csv", sep = ";")
df

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41183,73,retired,married,professional.course,no,yes,no,cellular,nov,fri,...,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes
41184,46,blue-collar,married,professional.course,no,no,no,cellular,nov,fri,...,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41185,56,retired,married,university.degree,no,yes,no,cellular,nov,fri,...,2,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41186,44,technician,married,professional.course,no,no,no,cellular,nov,fri,...,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes


time: 610 ms (started: 2026-04-14 07:09:21 +00:00)


In [8]:
# 3. Drop Leakage Feature
df.drop('duration', axis=1, inplace=True)


# 4. Handle "unknown" Values
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])


# 5. Encode Target
df['y'] = df['y'].map({'no': 0, 'yes': 1})

# 6. One-Hot Encoding
df = pd.get_dummies(df, drop_first=True)
X_shuffled = df.drop('y', axis=1)
y_shuffled = df['y']

# 7. Prepare your data
X_raw = df.drop('y', axis=1)
y_raw = df['y']

# 8. Apply SMOTE
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_raw, y_raw)

# 9. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_train_sm, y_train_sm, test_size=0.2, random_state=42
)


time: 485 ms (started: 2026-04-14 07:09:24 +00:00)


In [9]:
# Model
model =LogisticRegression(random_state=42, max_iter=1000, solver="liblinear", C = 1.0)

# Search space
param_dist = {
        'C': [0.1, 0.6, 0.8, 1],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'liblinear']
}

time: 659 µs (started: 2026-04-14 07:09:27 +00:00)


In [12]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_validate
search = RandomizedSearchCV(
      model,
      param_distributions=param_dist,
      n_iter=5,
      cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
      refit='roc_auc',
      n_jobs=-1,
      random_state=42,
      return_train_score=False
      )
search.fit(X_train, y_train)

best_model_LR = search.best_estimator_
print("Best parameters:", search.best_params_)
print("Best CV accuracy: {:.4f}".format(search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'solver': 'liblinear', 'penalty': 'l2', 'C': 0.1}
Best CV accuracy: 0.8557
Test accuracy: 0.8553
time: 3min 23s (started: 2026-04-14 07:11:49 +00:00)


In [16]:
def evaluate_models(name, X_shuffled, y_shuffled, label):

    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
      X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
      y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

      # Train
      start_time = time.time()
      model.fit(X_train, y_train)
      training_time = time.time() - start_time

      # Predict
      y_pred = model.predict(X_test)

      # Metrics
      acc = accuracy_score(y_test, y_pred)
      prec = precision_score(y_test, y_pred, zero_division=0)
      rec = recall_score(y_test, y_pred, zero_division=0)
      f1 = f1_score(y_test, y_pred, zero_division=0)
      cm = confusion_matrix(y_test, y_pred)
      tn, fp, fn, tp = cm.ravel()
      specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
      gm = np.sqrt(rec * specificity)
      auc = roc_auc_score(y_test, y_pred)
      mcc = matthews_corrcoef(y_test, y_pred)
      kappa = cohen_kappa_score(y_test, y_pred)
      balanced_acc = balanced_accuracy_score(y_test, y_pred)

      # Store fold results
      result.append({
          "Fold": fold+1,
          "Classifier": name,
          "Accuracy": acc,
          "Precision": prec,
          "Recall": rec,
          "Specificity": specificity,
          "F1": f1,
          "GM": gm,
          "AUC": auc,
          "MCC": mcc,
          "Kappa": kappa,
          "Balanced Accuracy": balanced_acc,
          "Training Time (s)": training_time
      })
    return pd.DataFrame(result)

time: 3.06 ms (started: 2026-04-14 07:21:34 +00:00)


In [17]:
result = []

time: 461 µs (started: 2026-04-14 07:21:36 +00:00)


In [18]:
model_name = "Logistic Regression"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)

   Fold           Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  Logistic Regression  0.859781   0.888073  0.824380     0.895416   
1     2  Logistic Regression  0.850068   0.874963  0.813587     0.885877   
2     3  Logistic Regression  0.856498   0.873304  0.832233     0.880512   
3     4  Logistic Regression  0.851847   0.872591  0.825803     0.878163   
4     5  Logistic Regression  0.858413   0.890464  0.819565     0.897796   
5     6  Logistic Regression  0.856908   0.879791  0.827417     0.886513   
6     7  Logistic Regression  0.855110   0.894474  0.812984     0.899327   
7     8  Logistic Regression  0.854563   0.876363  0.828587     0.881005   
8     9  Logistic Regression  0.860446   0.877723  0.834576     0.885846   
9    10  Logistic Regression  0.849501   0.870812  0.811077     0.885928   

         F1        GM       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.855042  0.859164  0.859898  0.721495  0.719625           0.859898   
1  0.843160  0.

In [19]:
# Model
from sklearn.tree import DecisionTreeClassifier
model1 =DecisionTreeClassifier()

# Search space
param_dist1 = {

        'max_depth': [None, 1, 3, 5],
        'min_samples_split': [2, 5, 10, 15],
        'min_samples_leaf': [1, 2, 5, 10],
        'criterion': ['gini', 'entropy']
}

time: 751 µs (started: 2026-04-14 07:24:57 +00:00)


In [21]:
from sklearn.model_selection import RandomizedSearchCV
# DecisionTreeClassifier

random_search = RandomizedSearchCV(model1, param_distributions=param_dist1, n_iter=50,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': None, 'criterion': 'entropy'}
Best CV accuracy: 0.8994
Test accuracy: 0.8979
time: 2min 8s (started: 2026-04-14 07:26:40 +00:00)


In [24]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_validate
search = RandomizedSearchCV(
      model,
      param_distributions=param_dist,
      n_iter=5,
      cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
      refit='roc_auc',
      n_jobs=-1,
      random_state=42,
      return_train_score=False
      )
search.fit(X_train, y_train)

best_model_LR = search.best_estimator_
print("Best parameters:", search.best_params_)
print("Best CV accuracy: {:.4f}".format(search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'solver': 'liblinear', 'penalty': 'l2', 'C': 0.1}
Best CV accuracy: 0.8557
Test accuracy: 0.8553
time: 3min 33s (started: 2026-04-14 07:32:10 +00:00)


In [25]:
def evaluate_models(name, X_shuffled, y_shuffled, label):

    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
      X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
      y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

      # Train
      start_time = time.time()
      model.fit(X_train, y_train)
      training_time = time.time() - start_time

      # Predict
      y_pred = model.predict(X_test)

      # Metrics
      acc = accuracy_score(y_test, y_pred)
      prec = precision_score(y_test, y_pred, zero_division=0)
      rec = recall_score(y_test, y_pred, zero_division=0)
      f1 = f1_score(y_test, y_pred, zero_division=0)
      cm = confusion_matrix(y_test, y_pred)
      tn, fp, fn, tp = cm.ravel()
      specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
      gm = np.sqrt(rec * specificity)
      auc = roc_auc_score(y_test, y_pred)
      mcc = matthews_corrcoef(y_test, y_pred)
      kappa = cohen_kappa_score(y_test, y_pred)
      balanced_acc = balanced_accuracy_score(y_test, y_pred)

      # Store fold results
      result.append({
          "Fold": fold+1,
          "Classifier": name,
          "Accuracy": acc,
          "Precision": prec,
          "Recall": rec,
          "Specificity": specificity,
          "F1": f1,
          "GM": gm,
          "AUC": auc,
          "MCC": mcc,
          "Kappa": kappa,
          "Balanced Accuracy": balanced_acc,
          "Training Time (s)": training_time
      })
    return pd.DataFrame(result)

time: 3.29 ms (started: 2026-04-14 07:36:12 +00:00)


In [26]:
result = []

time: 476 µs (started: 2026-04-14 07:36:16 +00:00)


In [27]:
model_name = "Decision Tree"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)

   Fold     Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1  Decision Tree  0.859781   0.888073  0.824380     0.895416  0.855042   
1     2  Decision Tree  0.850068   0.874963  0.813587     0.885877  0.843160   
2     3  Decision Tree  0.856498   0.873304  0.832233     0.880512  0.852274   
3     4  Decision Tree  0.851847   0.872591  0.825803     0.878163  0.848553   
4     5  Decision Tree  0.858413   0.890464  0.819565     0.897796  0.853545   
5     6  Decision Tree  0.856908   0.879791  0.827417     0.886513  0.852800   
6     7  Decision Tree  0.855110   0.894474  0.812984     0.899327  0.851784   
7     8  Decision Tree  0.854563   0.876363  0.828587     0.881005  0.851805   
8     9  Decision Tree  0.860446   0.877723  0.834576     0.885846  0.855606   
9    10  Decision Tree  0.849501   0.870812  0.811077     0.885928  0.839884   

         GM       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.859164  0.859898  0.721495  0.719625           0.8

In [28]:
# Model
from sklearn.ensemble import RandomForestClassifier
model2 = RandomForestClassifier()
# Search space
param_dist2 = {
        'n_estimators': [51, 101, 151],
        'max_depth': [None, 1, 3, 5],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 3],
        'max_features': ['sqrt', 'log2']
}

time: 1.3 ms (started: 2026-04-14 07:37:14 +00:00)


In [29]:

# RandomForestClassifier

random_search = RandomizedSearchCV(model2, param_distributions=param_dist2, n_iter=15,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'n_estimators': 51, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None}
Best CV accuracy: 0.9308
Test accuracy: 0.9288
time: 4min 52s (started: 2026-04-14 07:37:17 +00:00)


In [30]:
def evaluate_models(name, X_shuffled, y_shuffled, label):

    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
      X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
      y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

      # Train
      start_time = time.time()
      model.fit(X_train, y_train)
      training_time = time.time() - start_time

      # Predict
      y_pred = model.predict(X_test)

      # Metrics
      acc = accuracy_score(y_test, y_pred)
      prec = precision_score(y_test, y_pred, zero_division=0)
      rec = recall_score(y_test, y_pred, zero_division=0)
      f1 = f1_score(y_test, y_pred, zero_division=0)
      cm = confusion_matrix(y_test, y_pred)
      tn, fp, fn, tp = cm.ravel()
      specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
      gm = np.sqrt(rec * specificity)
      auc = roc_auc_score(y_test, y_pred)
      mcc = matthews_corrcoef(y_test, y_pred)
      kappa = cohen_kappa_score(y_test, y_pred)
      balanced_acc = balanced_accuracy_score(y_test, y_pred)

      # Store fold results
      result.append({
          "Fold": fold+1,
          "Classifier": name,
          "Accuracy": acc,
          "Precision": prec,
          "Recall": rec,
          "Specificity": specificity,
          "F1": f1,
          "GM": gm,
          "AUC": auc,
          "MCC": mcc,
          "Kappa": kappa,
          "Balanced Accuracy": balanced_acc,
          "Training Time (s)": training_time
      })
    return pd.DataFrame(result)


time: 2.98 ms (started: 2026-04-14 07:42:11 +00:00)


In [31]:
result=[]

time: 318 µs (started: 2026-04-14 07:42:11 +00:00)


In [32]:
model_name = "Random Forest"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)

   Fold     Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1  Random Forest  0.859781   0.888073  0.824380     0.895416  0.855042   
1     2  Random Forest  0.850068   0.874963  0.813587     0.885877  0.843160   
2     3  Random Forest  0.856498   0.873304  0.832233     0.880512  0.852274   
3     4  Random Forest  0.851847   0.872591  0.825803     0.878163  0.848553   
4     5  Random Forest  0.858413   0.890464  0.819565     0.897796  0.853545   
5     6  Random Forest  0.856908   0.879791  0.827417     0.886513  0.852800   
6     7  Random Forest  0.855110   0.894474  0.812984     0.899327  0.851784   
7     8  Random Forest  0.854563   0.876363  0.828587     0.881005  0.851805   
8     9  Random Forest  0.860446   0.877723  0.834576     0.885846  0.855606   
9    10  Random Forest  0.849501   0.870812  0.811077     0.885928  0.839884   

         GM       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.859164  0.859898  0.721495  0.719625           0.8

In [36]:

# Model Gradient Boosting

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, BaggingClassifier, StackingClassifier

model8 =   XGBClassifier(
         random_state=42,
         learning_rate=0.3,
         n_estimators=20
     )

param_dist8 = {
        'n_estimators': [21, 51, 81],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [1, 2, 3, 5],
        'subsample': [0.6, 0.8, 0.9]
}

time: 956 µs (started: 2026-04-14 07:58:19 +00:00)


In [40]:
from sklearn.model_selection import RandomizedSearchCV
# Gradient Boosting

random_search = RandomizedSearchCV(model8, param_distributions=param_dist8, n_iter=5,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'subsample': 0.9, 'n_estimators': 81, 'max_depth': 3, 'learning_rate': 0.05}
Best CV accuracy: 0.8246
Test accuracy: 0.8194
time: 27.7 s (started: 2026-04-14 08:07:40 +00:00)


In [94]:
def evaluate_models(name, X_shuffled, y_shuffled, label):

    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
      X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
      y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

      # Train
      start_time = time.time()
      model.fit(X_train, y_train)
      training_time = time.time() - start_time

      # Predict
      y_pred = model.predict(X_test)

      # Metrics
      acc = accuracy_score(y_test, y_pred)
      prec = precision_score(y_test, y_pred, zero_division=0)
      rec = recall_score(y_test, y_pred, zero_division=0)
      f1 = f1_score(y_test, y_pred, zero_division=0)
      cm = confusion_matrix(y_test, y_pred)
      tn, fp, fn, tp = cm.ravel()
      specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
      gm = np.sqrt(rec * specificity)
      auc = roc_auc_score(y_test, y_pred)
      mcc = matthews_corrcoef(y_test, y_pred)
      kappa = cohen_kappa_score(y_test, y_pred)
      balanced_acc = balanced_accuracy_score(y_test, y_pred)

      # Store fold results
      result.append({
          "Fold": fold+1,
          "Classifier": name,
          "Accuracy": acc,
          "Precision": prec,
          "Recall": rec,
          "Specificity": specificity,
          "F1": f1,
          "GM": gm,
          "AUC": auc,
          "MCC": mcc,
          "Kappa": kappa,
          "Balanced Accuracy": balanced_acc,
          "Training Time (s)": training_time
      })
    return pd.DataFrame(result)


time: 2.38 ms (started: 2026-04-14 09:26:09 +00:00)


In [95]:
result=[]

time: 421 µs (started: 2026-04-14 09:26:13 +00:00)


In [96]:
model_name = "Gradient Boosting (GBM)"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)

   Fold               Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  Gradient Boosting (GBM)  0.859781   0.888073  0.824380     0.895416   
1     2  Gradient Boosting (GBM)  0.850068   0.874963  0.813587     0.885877   
2     3  Gradient Boosting (GBM)  0.856498   0.873304  0.832233     0.880512   
3     4  Gradient Boosting (GBM)  0.851847   0.872591  0.825803     0.878163   
4     5  Gradient Boosting (GBM)  0.858413   0.890464  0.819565     0.897796   
5     6  Gradient Boosting (GBM)  0.856908   0.879791  0.827417     0.886513   
6     7  Gradient Boosting (GBM)  0.855110   0.894474  0.812984     0.899327   
7     8  Gradient Boosting (GBM)  0.854563   0.876363  0.828587     0.881005   
8     9  Gradient Boosting (GBM)  0.860446   0.877723  0.834576     0.885846   
9    10  Gradient Boosting (GBM)  0.849501   0.870812  0.811077     0.885928   

         F1        GM       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.855042  0.859164  0.859898  0.721495  0.

In [38]:


# Model

from xgboost import XGBClassifier

model5 =  XGBClassifier(
         random_state=42,
         n_estimators=100,
         max_depth=3,
         learning_rate=0.3,
      )

param_dist5 = {
        'n_estimators': [101, 201, 251],
        'learning_rate': [0.01, 0.05, 0.08],
        'max_depth': [3, 5, 6, 7],
        'subsample': [0.5, 0.8, 1.0],
        'colsample_bytree': [0.5, 0.8, 1.0]
}

time: 2.49 ms (started: 2026-04-14 08:00:18 +00:00)


In [41]:
from sklearn.model_selection import RandomizedSearchCV
# XGBoost

random_search = RandomizedSearchCV(model5, param_distributions=param_dist5, n_iter=5,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'subsample': 0.5, 'n_estimators': 101, 'max_depth': 3, 'learning_rate': 0.08, 'colsample_bytree': 0.8}
Best CV accuracy: 0.8596
Test accuracy: 0.8549
time: 1min 34s (started: 2026-04-14 08:09:31 +00:00)


In [42]:
def evaluate_models(name, X_shuffled, y_shuffled, label):

    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
      X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
      y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

      # Train
      start_time = time.time()
      model.fit(X_train, y_train)
      training_time = time.time() - start_time

      # Predict
      y_pred = model.predict(X_test)

      # Metrics
      acc = accuracy_score(y_test, y_pred)
      prec = precision_score(y_test, y_pred, zero_division=0)
      rec = recall_score(y_test, y_pred, zero_division=0)
      f1 = f1_score(y_test, y_pred, zero_division=0)
      cm = confusion_matrix(y_test, y_pred)
      tn, fp, fn, tp = cm.ravel()
      specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
      gm = np.sqrt(rec * specificity)
      auc = roc_auc_score(y_test, y_pred)
      mcc = matthews_corrcoef(y_test, y_pred)
      kappa = cohen_kappa_score(y_test, y_pred)
      balanced_acc = balanced_accuracy_score(y_test, y_pred)

      # Store fold results
      result.append({
          "Fold": fold+1,
          "Classifier": name,
          "Accuracy": acc,
          "Precision": prec,
          "Recall": rec,
          "Specificity": specificity,
          "F1": f1,
          "GM": gm,
          "AUC": auc,
          "MCC": mcc,
          "Kappa": kappa,
          "Balanced Accuracy": balanced_acc,
          "Training Time (s)": training_time
      })
    return pd.DataFrame(result)


time: 2.1 ms (started: 2026-04-14 08:11:43 +00:00)


In [43]:
result=[]

time: 426 µs (started: 2026-04-14 08:11:46 +00:00)


In [44]:
model_name = "XGBoost"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)

   Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1    XGBoost  0.859781   0.888073  0.824380     0.895416  0.855042   
1     2    XGBoost  0.850068   0.874963  0.813587     0.885877  0.843160   
2     3    XGBoost  0.856498   0.873304  0.832233     0.880512  0.852274   
3     4    XGBoost  0.851847   0.872591  0.825803     0.878163  0.848553   
4     5    XGBoost  0.858413   0.890464  0.819565     0.897796  0.853545   
5     6    XGBoost  0.856908   0.879791  0.827417     0.886513  0.852800   
6     7    XGBoost  0.855110   0.894474  0.812984     0.899327  0.851784   
7     8    XGBoost  0.854563   0.876363  0.828587     0.881005  0.851805   
8     9    XGBoost  0.860446   0.877723  0.834576     0.885846  0.855606   
9    10    XGBoost  0.849501   0.870812  0.811077     0.885928  0.839884   

         GM       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.859164  0.859898  0.721495  0.719625           0.859898   
1  0.848963  0.849732  0.701616  0.

In [45]:


# Model LightGBM



model6 =  XGBClassifier(
         max_depth=[3,5],
         learning_rate=0.3,
         n_estimators=100
     )

param_dist6 = {
        'n_estimators': [21, 51, 101],
        'learning_rate': [0.01, 0.05, 0.08],
        'num_leaves': [30, 50, 70],
        'max_depth': [-1, 5, 10],
        'subsample': [0.2, 0.5, 0.8],
        'colsample_bytree': [0.2, 0.5, 0.8]
}

time: 697 µs (started: 2026-04-14 08:12:01 +00:00)


In [46]:
from sklearn.model_selection import RandomizedSearchCV
# LightGBM

random_search = RandomizedSearchCV(model6, param_distributions=param_dist6, n_iter=5,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
20 fits failed out of a total of 50.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
20 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/xgboost/core.py", line 751, in inner_f
    return func(**kwargs)
           ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/xgboost/sklearn.py", line 1806, in fit
    self._Booster = train(
                    ^^^^^^
  File "/usr/local/lib/python3.12/di

Best parameters: {'subsample': 0.2, 'num_leaves': 50, 'n_estimators': 21, 'max_depth': 5, 'learning_rate': 0.08, 'colsample_bytree': 0.5}
Best CV accuracy: 0.8068
Test accuracy: 0.8055
time: 23.1 s (started: 2026-04-14 08:12:01 +00:00)


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [08:12:24] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "num_leaves" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [47]:
def evaluate_models(name, X_shuffled, y_shuffled, label):

    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
      X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
      y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

      # Train
      start_time = time.time()
      model.fit(X_train, y_train)
      training_time = time.time() - start_time

      # Predict
      y_pred = model.predict(X_test)

      # Metrics
      acc = accuracy_score(y_test, y_pred)
      prec = precision_score(y_test, y_pred, zero_division=0)
      rec = recall_score(y_test, y_pred, zero_division=0)
      f1 = f1_score(y_test, y_pred, zero_division=0)
      cm = confusion_matrix(y_test, y_pred)
      tn, fp, fn, tp = cm.ravel()
      specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
      gm = np.sqrt(rec * specificity)
      auc = roc_auc_score(y_test, y_pred)
      mcc = matthews_corrcoef(y_test, y_pred)
      kappa = cohen_kappa_score(y_test, y_pred)
      balanced_acc = balanced_accuracy_score(y_test, y_pred)

      # Store fold results
      result.append({
          "Fold": fold+1,
          "Classifier": name,
          "Accuracy": acc,
          "Precision": prec,
          "Recall": rec,
          "Specificity": specificity,
          "F1": f1,
          "GM": gm,
          "AUC": auc,
          "MCC": mcc,
          "Kappa": kappa,
          "Balanced Accuracy": balanced_acc,
          "Training Time (s)": training_time
      })
    return pd.DataFrame(result)


time: 3.02 ms (started: 2026-04-14 08:12:24 +00:00)


In [48]:
result=[]

time: 456 µs (started: 2026-04-14 08:12:24 +00:00)


In [49]:
model_name = "LightGBM"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)

   Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1   LightGBM  0.859781   0.888073  0.824380     0.895416  0.855042   
1     2   LightGBM  0.850068   0.874963  0.813587     0.885877  0.843160   
2     3   LightGBM  0.856498   0.873304  0.832233     0.880512  0.852274   
3     4   LightGBM  0.851847   0.872591  0.825803     0.878163  0.848553   
4     5   LightGBM  0.858413   0.890464  0.819565     0.897796  0.853545   
5     6   LightGBM  0.856908   0.879791  0.827417     0.886513  0.852800   
6     7   LightGBM  0.855110   0.894474  0.812984     0.899327  0.851784   
7     8   LightGBM  0.854563   0.876363  0.828587     0.881005  0.851805   
8     9   LightGBM  0.860446   0.877723  0.834576     0.885846  0.855606   
9    10   LightGBM  0.849501   0.870812  0.811077     0.885928  0.839884   

         GM       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.859164  0.859898  0.721495  0.719625           0.859898   
1  0.848963  0.849732  0.701616  0.

In [50]:


# Model CatBoost

from xgboost import XGBClassifier

model7 =   XGBClassifier(n_estimators=500,
        max_depth=3,
        learning_rate=0.1,)

param_dist7 = {
        'iterations': [50, 100, 200],
        'learning_rate': [0.01, 0.05, 0.1],
        'depth': [4, 6, 8],
        'l2_leaf_reg': [1, 3, 5]
}

time: 1 ms (started: 2026-04-14 08:12:36 +00:00)


In [51]:
from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(model7, param_distributions=param_dist7, n_iter=5,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [08:15:11] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "depth", "iterations", "l2_leaf_reg" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best parameters: {'learning_rate': 0.05, 'l2_leaf_reg': 3, 'iterations': 200, 'depth': 4}
Best CV accuracy: 0.8854
Test accuracy: 0.8798
time: 2min 37s (started: 2026-04-14 08:12:36 +00:00)


In [52]:
def evaluate_models(name, X_shuffled, y_shuffled, label):

    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
      X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
      y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

      # Train
      start_time = time.time()
      model.fit(X_train, y_train)
      training_time = time.time() - start_time

      # Predict
      y_pred = model.predict(X_test)

      # Metrics
      acc = accuracy_score(y_test, y_pred)
      prec = precision_score(y_test, y_pred, zero_division=0)
      rec = recall_score(y_test, y_pred, zero_division=0)
      f1 = f1_score(y_test, y_pred, zero_division=0)
      cm = confusion_matrix(y_test, y_pred)
      tn, fp, fn, tp = cm.ravel()
      specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
      gm = np.sqrt(rec * specificity)
      auc = roc_auc_score(y_test, y_pred)
      mcc = matthews_corrcoef(y_test, y_pred)
      kappa = cohen_kappa_score(y_test, y_pred)
      balanced_acc = balanced_accuracy_score(y_test, y_pred)

      # Store fold results
      result.append({
          "Fold": fold+1,
          "Classifier": name,
          "Accuracy": acc,
          "Precision": prec,
          "Recall": rec,
          "Specificity": specificity,
          "F1": f1,
          "GM": gm,
          "AUC": auc,
          "MCC": mcc,
          "Kappa": kappa,
          "Balanced Accuracy": balanced_acc,
          "Training Time (s)": training_time
      })
    return pd.DataFrame(result)


time: 3.14 ms (started: 2026-04-14 08:15:14 +00:00)


In [53]:
result=[]

time: 1.19 ms (started: 2026-04-14 08:15:14 +00:00)


In [54]:
model_name = "CatBoost"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)

   Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1   CatBoost  0.859781   0.888073  0.824380     0.895416  0.855042   
1     2   CatBoost  0.850068   0.874963  0.813587     0.885877  0.843160   
2     3   CatBoost  0.856498   0.873304  0.832233     0.880512  0.852274   
3     4   CatBoost  0.851847   0.872591  0.825803     0.878163  0.848553   
4     5   CatBoost  0.858413   0.890464  0.819565     0.897796  0.853545   
5     6   CatBoost  0.856908   0.879791  0.827417     0.886513  0.852800   
6     7   CatBoost  0.855110   0.894474  0.812984     0.899327  0.851784   
7     8   CatBoost  0.854563   0.876363  0.828587     0.881005  0.851805   
8     9   CatBoost  0.860446   0.877723  0.834576     0.885846  0.855606   
9    10   CatBoost  0.849501   0.870812  0.811077     0.885928  0.839884   

         GM       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.859164  0.859898  0.721495  0.719625           0.859898   
1  0.848963  0.849732  0.701616  0.

In [55]:
# Model
from sklearn.svm import LinearSVC
model4 = LinearSVC()



from sklearn.svm import LinearSVC

model4 = LinearSVC(random_state=42, C=1.0,tol=1e-3, max_iter=2000)

param_dist4 = {

        'C': [10, 50, 70]
}

time: 807 µs (started: 2026-04-14 08:15:25 +00:00)


In [56]:
from sklearn.model_selection import RandomizedSearchCV
# LinearSVC

random_search = RandomizedSearchCV(model4, param_distributions=param_dist4, n_iter=5,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 3 is smaller than n_iter=5. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best parameters: {'C': 70}
Best CV accuracy: 0.8527
Test accuracy: 0.8531
time: 25.2 s (started: 2026-04-14 08:15:25 +00:00)


In [57]:
def evaluate_models(name, X_shuffled, y_shuffled, label):

    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
      X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
      y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

      # Train
      start_time = time.time()
      model.fit(X_train, y_train)
      training_time = time.time() - start_time

      # Predict
      y_pred = model.predict(X_test)

      # Metrics
      acc = accuracy_score(y_test, y_pred)
      prec = precision_score(y_test, y_pred, zero_division=0)
      rec = recall_score(y_test, y_pred, zero_division=0)
      f1 = f1_score(y_test, y_pred, zero_division=0)
      cm = confusion_matrix(y_test, y_pred)
      tn, fp, fn, tp = cm.ravel()
      specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
      gm = np.sqrt(rec * specificity)
      auc = roc_auc_score(y_test, y_pred)
      mcc = matthews_corrcoef(y_test, y_pred)
      kappa = cohen_kappa_score(y_test, y_pred)
      balanced_acc = balanced_accuracy_score(y_test, y_pred)

      # Store fold results
      result.append({
          "Fold": fold+1,
          "Classifier": name,
          "Accuracy": acc,
          "Precision": prec,
          "Recall": rec,
          "Specificity": specificity,
          "F1": f1,
          "GM": gm,
          "AUC": auc,
          "MCC": mcc,
          "Kappa": kappa,
          "Balanced Accuracy": balanced_acc,
          "Training Time (s)": training_time
      })
    return pd.DataFrame(result)


time: 3.11 ms (started: 2026-04-14 08:15:50 +00:00)


In [58]:
result=[]

time: 399 µs (started: 2026-04-14 08:15:50 +00:00)


In [59]:
model_name = "Support Vector Machine"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)

   Fold              Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  Support Vector Machine  0.859781   0.888073  0.824380     0.895416   
1     2  Support Vector Machine  0.850068   0.874963  0.813587     0.885877   
2     3  Support Vector Machine  0.856498   0.873304  0.832233     0.880512   
3     4  Support Vector Machine  0.851847   0.872591  0.825803     0.878163   
4     5  Support Vector Machine  0.858413   0.890464  0.819565     0.897796   
5     6  Support Vector Machine  0.856908   0.879791  0.827417     0.886513   
6     7  Support Vector Machine  0.855110   0.894474  0.812984     0.899327   
7     8  Support Vector Machine  0.854563   0.876363  0.828587     0.881005   
8     9  Support Vector Machine  0.860446   0.877723  0.834576     0.885846   
9    10  Support Vector Machine  0.849501   0.870812  0.811077     0.885928   

         F1        GM       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.855042  0.859164  0.859898  0.721495  0.719625     

In [60]:
# Model
from sklearn.neighbors import KNeighborsClassifier
model3 = KNeighborsClassifier(
         n_neighbors=15,
         weights='distance')

# Search space
param_dist3 = {
        'n_neighbors': [3, 5, 7],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan']
}

time: 1.04 ms (started: 2026-04-14 08:16:05 +00:00)


In [61]:
from sklearn.model_selection import RandomizedSearchCV
# KNeighbors Classifier

random_search = RandomizedSearchCV(model3, param_distributions=param_dist3, n_iter=5,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'weights': 'distance', 'n_neighbors': 5, 'metric': 'manhattan'}
Best CV accuracy: 0.9179
Test accuracy: 0.9146
time: 8min 34s (started: 2026-04-14 08:16:05 +00:00)


In [62]:
def evaluate_models(name, X_shuffled, y_shuffled, label):

    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
      X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
      y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

      # Train
      start_time = time.time()
      model.fit(X_train, y_train)
      training_time = time.time() - start_time

      # Predict
      y_pred = model.predict(X_test)

      # Metrics
      acc = accuracy_score(y_test, y_pred)
      prec = precision_score(y_test, y_pred, zero_division=0)
      rec = recall_score(y_test, y_pred, zero_division=0)
      f1 = f1_score(y_test, y_pred, zero_division=0)
      cm = confusion_matrix(y_test, y_pred)
      tn, fp, fn, tp = cm.ravel()
      specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
      gm = np.sqrt(rec * specificity)
      auc = roc_auc_score(y_test, y_pred)
      mcc = matthews_corrcoef(y_test, y_pred)
      kappa = cohen_kappa_score(y_test, y_pred)
      balanced_acc = balanced_accuracy_score(y_test, y_pred)

      # Store fold results
      result.append({
          "Fold": fold+1,
          "Classifier": name,
          "Accuracy": acc,
          "Precision": prec,
          "Recall": rec,
          "Specificity": specificity,
          "F1": f1,
          "GM": gm,
          "AUC": auc,
          "MCC": mcc,
          "Kappa": kappa,
          "Balanced Accuracy": balanced_acc,
          "Training Time (s)": training_time
      })
    return pd.DataFrame(result)


time: 2.32 ms (started: 2026-04-14 08:24:40 +00:00)


In [63]:
result=[]

time: 429 µs (started: 2026-04-14 08:24:40 +00:00)


In [64]:
model_name = "K Nearest Neighbour"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)

   Fold           Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  K Nearest Neighbour  0.859781   0.888073  0.824380     0.895416   
1     2  K Nearest Neighbour  0.850068   0.874963  0.813587     0.885877   
2     3  K Nearest Neighbour  0.856498   0.873304  0.832233     0.880512   
3     4  K Nearest Neighbour  0.851847   0.872591  0.825803     0.878163   
4     5  K Nearest Neighbour  0.858413   0.890464  0.819565     0.897796   
5     6  K Nearest Neighbour  0.856908   0.879791  0.827417     0.886513   
6     7  K Nearest Neighbour  0.855110   0.894474  0.812984     0.899327   
7     8  K Nearest Neighbour  0.854563   0.876363  0.828587     0.881005   
8     9  K Nearest Neighbour  0.860446   0.877723  0.834576     0.885846   
9    10  K Nearest Neighbour  0.849501   0.870812  0.811077     0.885928   

         F1        GM       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.855042  0.859164  0.859898  0.721495  0.719625           0.859898   
1  0.843160  0.

In [65]:


# Model MLP

from sklearn.neural_network import MLPClassifier
model9 =   MLPClassifier(
         random_state=42,
         hidden_layer_sizes=(64,32),
         activation='relu',
         solver='adam',
         max_iter=200,
         early_stopping=True
         )
param_dist9 = {
      'hidden_layer_sizes': [(50,), (70,), (50,50)],
        'activation': ['relu', 'tanh'],
        'learning_rate': ['constant', 'adaptive'],
        'max_iter': [100, 300, 500]
}

time: 1.7 ms (started: 2026-04-14 08:24:51 +00:00)


In [66]:
from sklearn.model_selection import RandomizedSearchCV
# MLP

random_search = RandomizedSearchCV(model9, param_distributions=param_dist9, n_iter=5,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'max_iter': 300, 'learning_rate': 'constant', 'hidden_layer_sizes': (50, 50), 'activation': 'relu'}
Best CV accuracy: 0.8247
Test accuracy: 0.8326
time: 6min 20s (started: 2026-04-14 08:24:51 +00:00)


In [67]:
def evaluate_models(name, X_shuffled, y_shuffled, label):

    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
      X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
      y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

      # Train
      start_time = time.time()
      model.fit(X_train, y_train)
      training_time = time.time() - start_time

      # Predict
      y_pred = model.predict(X_test)

      # Metrics
      acc = accuracy_score(y_test, y_pred)
      prec = precision_score(y_test, y_pred, zero_division=0)
      rec = recall_score(y_test, y_pred, zero_division=0)
      f1 = f1_score(y_test, y_pred, zero_division=0)
      cm = confusion_matrix(y_test, y_pred)
      tn, fp, fn, tp = cm.ravel()
      specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
      gm = np.sqrt(rec * specificity)
      auc = roc_auc_score(y_test, y_pred)
      mcc = matthews_corrcoef(y_test, y_pred)
      kappa = cohen_kappa_score(y_test, y_pred)
      balanced_acc = balanced_accuracy_score(y_test, y_pred)

      # Store fold results
      result.append({
          "Fold": fold+1,
          "Classifier": name,
          "Accuracy": acc,
          "Precision": prec,
          "Recall": rec,
          "Specificity": specificity,
          "F1": f1,
          "GM": gm,
          "AUC": auc,
          "MCC": mcc,
          "Kappa": kappa,
          "Balanced Accuracy": balanced_acc,
          "Training Time (s)": training_time
      })
    return pd.DataFrame(result)


time: 5.52 ms (started: 2026-04-14 08:31:11 +00:00)


In [68]:
result=[]

time: 432 µs (started: 2026-04-14 08:31:11 +00:00)


In [69]:
model_name = "Multi Layer Perceptron"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)

   Fold              Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  Multi Layer Perceptron  0.859781   0.888073  0.824380     0.895416   
1     2  Multi Layer Perceptron  0.850068   0.874963  0.813587     0.885877   
2     3  Multi Layer Perceptron  0.856498   0.873304  0.832233     0.880512   
3     4  Multi Layer Perceptron  0.851847   0.872591  0.825803     0.878163   
4     5  Multi Layer Perceptron  0.858413   0.890464  0.819565     0.897796   
5     6  Multi Layer Perceptron  0.856908   0.879791  0.827417     0.886513   
6     7  Multi Layer Perceptron  0.855110   0.894474  0.812984     0.899327   
7     8  Multi Layer Perceptron  0.854563   0.876363  0.828587     0.881005   
8     9  Multi Layer Perceptron  0.860446   0.877723  0.834576     0.885846   
9    10  Multi Layer Perceptron  0.849501   0.870812  0.811077     0.885928   

         F1        GM       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.855042  0.859164  0.859898  0.721495  0.719625     

In [72]:
# Model Bagging
from sklearn.ensemble import BaggingClassifier
model_bagging =   BaggingClassifier(
         random_state=42,
         n_estimators=200,
         max_samples=0.8,
         max_features=0.8
         )
param_dist10 = {
     'n_estimators': [11, 21, 51],
        'max_samples': [0.5, 0.7, 1.0],
        'max_features': [0.3, 0.5, 0.7]
}

time: 999 µs (started: 2026-04-14 08:42:59 +00:00)


In [73]:
from sklearn.model_selection import RandomizedSearchCV
# BaggingClassifier

random_search_bagging = RandomizedSearchCV(model_bagging, param_distributions=param_dist10, n_iter=5,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search_bagging.fit(X_train, y_train)

best_model_bagging = random_search_bagging.best_estimator_
print("Best parameters:", random_search_bagging.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search_bagging.best_score_))
print("Test accuracy: {:.4f}".format(best_model_bagging.score(X_test, y_test)))

Best parameters: {'n_estimators': 11, 'max_samples': 0.7, 'max_features': 0.7}
Best CV accuracy: 0.9316
Test accuracy: 0.9310
time: 1min 34s (started: 2026-04-14 08:43:02 +00:00)


In [74]:
def evaluate_models(name, X_shuffled, y_shuffled, label):

    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
      X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
      y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

      # Train
      start_time = time.time()
      model.fit(X_train, y_train)
      training_time = time.time() - start_time

      # Predict
      y_pred = model.predict(X_test)

      # Metrics
      acc = accuracy_score(y_test, y_pred)
      prec = precision_score(y_test, y_pred, zero_division=0)
      rec = recall_score(y_test, y_pred, zero_division=0)
      f1 = f1_score(y_test, y_pred, zero_division=0)
      cm = confusion_matrix(y_test, y_pred)
      tn, fp, fn, tp = cm.ravel()
      specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
      gm = np.sqrt(rec * specificity)
      auc = roc_auc_score(y_test, y_pred)
      mcc = matthews_corrcoef(y_test, y_pred)
      kappa = cohen_kappa_score(y_test, y_pred)
      balanced_acc = balanced_accuracy_score(y_test, y_pred)

      # Store fold results
      result.append({
          "Fold": fold+1,
          "Classifier": name,
          "Accuracy": acc,
          "Precision": prec,
          "Recall": rec,
          "Specificity": specificity,
          "F1": f1,
          "GM": gm,
          "AUC": auc,
          "MCC": mcc,
          "Kappa": kappa,
          "Balanced Accuracy": balanced_acc,
          "Training Time (s)": training_time
      })
    return pd.DataFrame(result)


time: 2.38 ms (started: 2026-04-14 08:44:37 +00:00)


In [75]:
result=[]

time: 406 µs (started: 2026-04-14 08:44:37 +00:00)


In [76]:
model_name = "Bagging Classifier"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)

   Fold          Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  Bagging Classifier  0.859781   0.888073  0.824380     0.895416   
1     2  Bagging Classifier  0.850068   0.874963  0.813587     0.885877   
2     3  Bagging Classifier  0.856498   0.873304  0.832233     0.880512   
3     4  Bagging Classifier  0.851847   0.872591  0.825803     0.878163   
4     5  Bagging Classifier  0.858413   0.890464  0.819565     0.897796   
5     6  Bagging Classifier  0.856908   0.879791  0.827417     0.886513   
6     7  Bagging Classifier  0.855110   0.894474  0.812984     0.899327   
7     8  Bagging Classifier  0.854563   0.876363  0.828587     0.881005   
8     9  Bagging Classifier  0.860446   0.877723  0.834576     0.885846   
9    10  Bagging Classifier  0.849501   0.870812  0.811077     0.885928   

         F1        GM       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.855042  0.859164  0.859898  0.721495  0.719625           0.859898   
1  0.843160  0.848963  0.8

In [84]:
# Model Stacking
from sklearn.ensemble import StackingClassifier
model_stacking =   StackingClassifier(
          cv=3,
        estimators=[
            ('lr', LogisticRegression(random_state=42)),
            ('rf', RandomForestClassifier(random_state=42,
                                          n_estimators=50,
                                          max_depth=10,
                                          n_jobs=-1)),
            ('dt', DecisionTreeClassifier(random_state=42, max_depth=10)),
        ],
        n_jobs=-1,
        passthrough=False,
        verbose=1,
         )
param_dist11 = {
       'passthrough': [False],
        'final_estimator': [LogisticRegression(random_state=42, max_iter=200)]
}

time: 2.86 ms (started: 2026-04-14 09:14:01 +00:00)


In [85]:
from sklearn.model_selection import RandomizedSearchCV
# StackingClassifier

random_search_stacking = RandomizedSearchCV(model_stacking, param_distributions=param_dist11, n_iter=1,
                                   cv=3, scoring="accuracy", random_state=42, n_jobs=-1)
random_search_stacking.fit(X_train, y_train)

best_model_stacking = random_search_stacking.best_estimator_
print("Best parameters:", random_search_stacking.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search_stacking.best_score_))
print("Test accuracy: {:.4f}".format(best_model_stacking.score(X_test, y_test)))

Best parameters: {'passthrough': False, 'final_estimator': LogisticRegression(max_iter=200, random_state=42)}
Best CV accuracy: 0.8441
Test accuracy: 0.8435
time: 37 s (started: 2026-04-14 09:14:04 +00:00)


In [86]:
def evaluate_models(name, X_shuffled, y_shuffled, label):

    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
      X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
      y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

      # Train
      start_time = time.time()
      model.fit(X_train, y_train)
      training_time = time.time() - start_time

      # Predict
      y_pred = model.predict(X_test)

      # Metrics
      acc = accuracy_score(y_test, y_pred)
      prec = precision_score(y_test, y_pred, zero_division=0)
      rec = recall_score(y_test, y_pred, zero_division=0)
      f1 = f1_score(y_test, y_pred, zero_division=0)
      cm = confusion_matrix(y_test, y_pred)
      tn, fp, fn, tp = cm.ravel()
      specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
      gm = np.sqrt(rec * specificity)
      auc = roc_auc_score(y_test, y_pred)
      mcc = matthews_corrcoef(y_test, y_pred)
      kappa = cohen_kappa_score(y_test, y_pred)
      balanced_acc = balanced_accuracy_score(y_test, y_pred)

      # Store fold results
      result.append({
          "Fold": fold+1,
          "Classifier": name,
          "Accuracy": acc,
          "Precision": prec,
          "Recall": rec,
          "Specificity": specificity,
          "F1": f1,
          "GM": gm,
          "AUC": auc,
          "MCC": mcc,
          "Kappa": kappa,
          "Balanced Accuracy": balanced_acc,
          "Training Time (s)": training_time
      })
    return pd.DataFrame(result)


time: 2.35 ms (started: 2026-04-14 09:14:46 +00:00)


In [87]:
result=[]

time: 420 µs (started: 2026-04-14 09:14:52 +00:00)


In [88]:
model_name = "Stacking Classifier"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)

   Fold           Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  Stacking Classifier  0.859781   0.888073  0.824380     0.895416   
1     2  Stacking Classifier  0.850068   0.874963  0.813587     0.885877   
2     3  Stacking Classifier  0.856498   0.873304  0.832233     0.880512   
3     4  Stacking Classifier  0.851847   0.872591  0.825803     0.878163   
4     5  Stacking Classifier  0.858413   0.890464  0.819565     0.897796   
5     6  Stacking Classifier  0.856908   0.879791  0.827417     0.886513   
6     7  Stacking Classifier  0.855110   0.894474  0.812984     0.899327   
7     8  Stacking Classifier  0.854563   0.876363  0.828587     0.881005   
8     9  Stacking Classifier  0.860446   0.877723  0.834576     0.885846   
9    10  Stacking Classifier  0.849501   0.870812  0.811077     0.885928   

         F1        GM       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.855042  0.859164  0.859898  0.721495  0.719625           0.859898   
1  0.843160  0.